# Analisi Esplorativa del Dataset

Questo notebook esegue:
1. **Controllo qualità dei dati** - Verifica valori mancanti, duplicati, outlier, tipi di dati
2. **Analisi esplorativa completa** - Estrazione features da tutti i file
3. **Selezione utenti rappresentativi** - Identificazione dei 3 utenti più distinti per l'analisi approfondita

> **Nota metodologica:** Il notebook esegue solo controlli e analisi esplorativa, senza modificare i dati originali. Gli utenti selezionati saranno poi utilizzati in `analisi_segnali.ipynb` per l'analisi dettagliata.

---

## 1. Importazione delle librerie necessarie
Importiamo le librerie fondamentali per la manipolazione e l'analisi dei dati.

In [8]:
import os
import pandas as pd
import numpy as np
from glob import glob

data_dir = 'RawData'
file_paths = sorted(glob(os.path.join(data_dir, 'gyro_exp*.txt')))
label_path = os.path.join(data_dir, 'labels.txt')

## 2. Caricamento dei dati di esempio
Carichiamo un file di esempio e il file delle label per eseguire i controlli.

In [9]:
sample_data_path = file_paths[0] if file_paths else None
if sample_data_path:
    df_sample = pd.read_csv(sample_data_path, sep=' ', header=None)
    print(f"Caricato: {sample_data_path}")
    display(df_sample.head())
else:
    print("Nessun file di dati trovato.")

df_labels = pd.read_csv(label_path, sep=' ', header=None)
print(f"Caricate {len(df_labels)} etichette da labels.txt")
display(df_labels.head())

Caricato: RawData/gyro_exp01_user01.txt


,0,1,2
0,-0.054978,-0.069639,-0.030849
1,-0.012523,0.019242,-0.038485
2,-0.023518,0.276417,0.006414
3,-0.093462,0.367741,0.001222
4,-0.124311,0.476780,-0.022907


Caricate 412 etichette da labels.txt


,0,1,2,3,4
0,1,1,5,250,1232
1,1,1,7,1233,1392
2,1,1,4,1393,2194
3,1,1,8,2195,2359
4,1,1,5,2360,3374


## 3. Controllo dei valori mancanti
Verifichiamo la presenza di valori mancanti nei dati di esempio e nelle label.

In [10]:
dati_null = df_sample.isnull().sum() if sample_data_path else None
print("Valori mancanti nel file di esempio:")
print(dati_null)

label_null = df_labels.isnull().sum()
print("Valori mancanti nelle label:")
print(label_null)

Valori mancanti nel file di esempio:
0    0
1    0
2    0
dtype: int64
Valori mancanti nelle label:
0    0
1    0
2    0
3    0
4    0
dtype: int64


## 4. Controllo dei tipi di dati
Visualizziamo i tipi di dati delle colonne e verifichiamo la coerenza.

In [11]:
df_types = df_sample.dtypes if sample_data_path else None
print("Tipi di dati nel file di esempio:")
print(df_types)

print("Tipi di dati nelle label:")
print(df_labels.dtypes)

Tipi di dati nel file di esempio:
0    float64
1    float64
2    float64
dtype: object
Tipi di dati nelle label:
0    int64
1    int64
2    int64
3    int64
4    int64
dtype: object


## 5. Analisi Esplorativa Completa del Dataset

Analizziamo tutti i file per estrarre features statistiche e spettrali, e identificare i 3 utenti più distinti per l'analisi approfondita.

In [ ]:
# Importa librerie aggiuntive per l'analisi esplorativa
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from scipy.spatial.distance import pdist, squareform

# Configurazione
plt.style.use('seaborn-v0_8-darkgrid')
SAMPLING_FREQ = 50  # Hz

def extract_features(file_path):
    """Estrae features statistiche e spettrali da un file"""
    data = pd.read_csv(file_path, sep=' ', header=None, names=['x', 'y', 'z'])
    
    features = {}
    for axis in ['x', 'y', 'z']:
        signal = data[axis].values
        
        # Features temporali
        features[f'mean_{axis}'] = np.mean(signal)
        features[f'std_{axis}'] = np.std(signal)
        features[f'energy_{axis}'] = np.sum(signal**2)
        features[f'range_{axis}'] = np.max(signal) - np.min(signal)
        
        # Features spettrali
        fft_vals = np.fft.fft(signal)
        power_spectrum = np.abs(fft_vals[:len(fft_vals)//2])**2
        features[f'spectral_peak_{axis}'] = np.max(power_spectrum)
        features[f'spectral_mean_{axis}'] = np.mean(power_spectrum)
    
    return features

# Analizza tutti i file
print(f"Trovati {len(file_paths)} file da analizzare...\n")

# Crea un DataFrame con tutte le features
all_features = []
for file_path in file_paths:
    filename = Path(file_path).name
    exp_id = int(filename.split('_')[1].replace('exp', ''))
    user_id = int(filename.split('_')[2].replace('user', '').replace('.txt', ''))
    
    features = extract_features(file_path)
    features['exp_id'] = exp_id
    features['user_id'] = user_id
    features['filename'] = filename
    
    all_features.append(features)

features_df = pd.DataFrame(all_features)
print("Features estratte da tutti i file:")
print(features_df[['filename', 'user_id', 'exp_id', 'energy_x', 'std_x', 'range_x']].head(10))

## 6. Identificazione dei 3 Utenti più Distinti

Selezioniamo i 3 utenti con i pattern più differenziati analizzando:
1. Variabilità interna (quanto variano i loro esperimenti)
2. Distanza tra utenti (quanto sono diversi gli uni dagli altri)

In [ ]:
# Raggruppa per utente e calcola metriche aggregate
user_stats = features_df.groupby('user_id').agg({
    'energy_x': ['mean', 'std'],
    'energy_y': ['mean', 'std'],
    'energy_z': ['mean', 'std'],
    'std_x': ['mean', 'std'],
    'std_y': ['mean', 'std'],
    'std_z': ['mean', 'std'],
    'range_x': ['mean', 'std'],
    'range_y': ['mean', 'std'],
    'range_z': ['mean', 'std']
})

# Calcola un indice di "distintività" (variabilità interna normalizzata)
user_stats.columns = ['_'.join(col) for col in user_stats.columns]
user_stats['distinctiveness'] = (
    user_stats['energy_x_std'] / (user_stats['energy_x_mean'] + 1e-10) +
    user_stats['std_x_std'] / (user_stats['std_x_mean'] + 1e-10) +
    user_stats['range_x_std'] / (user_stats['range_x_mean'] + 1e-10)
) / 3

user_stats = user_stats.reset_index()
user_stats = user_stats.sort_values('distinctiveness', ascending=False)

print("Top utenti per distintività (variabilità interna tra esperimenti):")
print(user_stats[['user_id', 'distinctiveness', 'energy_x_mean', 'std_x_mean']].head())

# Calcoliamo la distanza tra utenti per trovare quelli più diversi tra loro
feature_cols = [col for col in user_stats.columns if col not in ['user_id', 'distinctiveness']]
X = user_stats[feature_cols].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Matrice delle distanze
distances = squareform(pdist(X_scaled, metric='euclidean'))
distance_df = pd.DataFrame(distances, 
                           index=user_stats['user_id'].values,
                           columns=user_stats['user_id'].values)

print("\nMatrice delle distanze tra utenti (primi 5x5):")
print(distance_df.iloc[:5, :5].round(2))

In [ ]:
# Algoritmo di selezione: massimizzare la diversità tra i 3 utenti
selected_users = []

# 1. Utente con maggiore distintività interna
first_user = user_stats.iloc[0]['user_id']
selected_users.append(int(first_user))

# 2. Utente più distante dal primo
distances_from_first = distance_df.loc[first_user]
second_user = distances_from_first[distances_from_first.index != first_user].idxmax()
selected_users.append(int(second_user))

# 3. Utente con massima distanza minima dai primi due
remaining_users = [u for u in distance_df.index if u not in selected_users]
max_min_distance = -1
third_user = None

for user in remaining_users:
    min_distance = min(distance_df.loc[user, selected_users[0]], 
                      distance_df.loc[user, selected_users[1]])
    if min_distance > max_min_distance:
        max_min_distance = min_distance
        third_user = user

selected_users.append(int(third_user))

print(f"\n{'='*60}")
print(f"✅ UTENTI SELEZIONATI PER L'ANALISI: {selected_users}")
print(f"{'='*60}\n")

for user_id in selected_users:
    user_data = features_df[features_df['user_id'] == user_id]
    print(f"User {user_id}:")
    print(f"  Esperimenti: {len(user_data)}")
    print(f"  Energia media (X): {user_data['energy_x'].mean():.2f}")
    print(f"  Std media (X): {user_data['std_x'].mean():.4f}")
    print(f"  Range medio (X): {user_data['range_x'].mean():.4f}\n")

# Visualizzazione della separazione
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for user_id in selected_users:
    user_data = features_df[features_df['user_id'] == user_id]
    axes[0].scatter(user_data['energy_x'], user_data['std_x'], 
                   s=100, label=f'User {user_id}', alpha=0.7)
    axes[1].scatter(user_data['range_x'], user_data['spectral_peak_x'], 
                   s=100, label=f'User {user_id}', alpha=0.7)

axes[0].set_xlabel('Energia (asse X)')
axes[0].set_ylabel('Deviazione Standard (asse X)')
axes[0].set_title('Separazione Temporale tra Utenti')
axes[0].legend()
axes[0].grid(True)

axes[1].set_xlabel('Range (asse X)')
axes[1].set_ylabel('Picco Spettrale (asse X)')
axes[1].set_title('Separazione Spettrale tra Utenti')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 7. Caricamento Dati degli Utenti Selezionati

Carichiamo i primi due esperimenti di ciascun utente selezionato per confronto e analisi dettagliata.

In [ ]:
# Carica i dati dei 3 utenti selezionati (primi 2 esperimenti ciascuno)
selected_data = {}

for user_id in selected_users:
    user_files = features_df[features_df['user_id'] == user_id].sort_values('exp_id')
    
    if len(user_files) >= 2:
        exp1_file = Path(data_dir) / user_files.iloc[0]['filename']
        exp2_file = Path(data_dir) / user_files.iloc[1]['filename']
        
        exp1_data = pd.read_csv(exp1_file, sep=' ', header=None, names=['x', 'y', 'z'])
        exp2_data = pd.read_csv(exp2_file, sep=' ', header=None, names=['x', 'y', 'z'])
        
        selected_data[user_id] = {
            'exp1': exp1_data,
            'exp2': exp2_data,
            'exp1_file': user_files.iloc[0]['filename'],
            'exp2_file': user_files.iloc[1]['filename']
        }

print("✅ Dati caricati per gli utenti selezionati:\n")
for user_id, data in selected_data.items():
    print(f"User {user_id}:")
    print(f"  Exp 1: {data['exp1_file']} - {len(data['exp1'])} campioni ({len(data['exp1'])/SAMPLING_FREQ:.1f}s)")
    print(f"  Exp 2: {data['exp2_file']} - {len(data['exp2'])} campioni ({len(data['exp2'])/SAMPLING_FREQ:.1f}s)\n")

# Visualizzazione comparativa dei segnali
fig, axes = plt.subplots(3, 2, figsize=(16, 10))
fig.suptitle('Confronto Segnali tra Utenti Selezionati', fontsize=16, y=1.00)

for idx, user_id in enumerate(selected_users):
    # Esperimento 1
    time1 = np.arange(len(selected_data[user_id]['exp1'])) / SAMPLING_FREQ
    segment1 = slice(0, min(500, len(selected_data[user_id]['exp1'])))
    
    axes[idx, 0].plot(time1[segment1], selected_data[user_id]['exp1']['x'].values[segment1], 
                     label='X', alpha=0.7, linewidth=1)
    axes[idx, 0].plot(time1[segment1], selected_data[user_id]['exp1']['y'].values[segment1], 
                     label='Y', alpha=0.7, linewidth=1)
    axes[idx, 0].plot(time1[segment1], selected_data[user_id]['exp1']['z'].values[segment1], 
                     label='Z', alpha=0.7, linewidth=1)
    axes[idx, 0].set_title(f'User {user_id} - Exp 1')
    axes[idx, 0].set_ylabel('Vel. ang. [rad/s]')
    axes[idx, 0].legend()
    axes[idx, 0].grid(True, alpha=0.3)
    
    # Esperimento 2
    time2 = np.arange(len(selected_data[user_id]['exp2'])) / SAMPLING_FREQ
    segment2 = slice(0, min(500, len(selected_data[user_id]['exp2'])))
    
    axes[idx, 1].plot(time2[segment2], selected_data[user_id]['exp2']['x'].values[segment2], 
                     label='X', alpha=0.7, linewidth=1)
    axes[idx, 1].plot(time2[segment2], selected_data[user_id]['exp2']['y'].values[segment2], 
                     label='Y', alpha=0.7, linewidth=1)
    axes[idx, 1].plot(time2[segment2], selected_data[user_id]['exp2']['z'].values[segment2], 
                     label='Z', alpha=0.7, linewidth=1)
    axes[idx, 1].set_title(f'User {user_id} - Exp 2')
    axes[idx, 1].legend()
    axes[idx, 1].grid(True, alpha=0.3)

axes[-1, 0].set_xlabel('Tempo [s]')
axes[-1, 1].set_xlabel('Tempo [s]')

plt.tight_layout()
plt.show()

print("\n" + "="*60)
print("Dataset pronto per l'analisi approfondita in 'analisi_segnali.ipynb'")
print("="*60)